# FAISS Data Insertion
This notebook generates embeddings for atomic queries using the `all-MiniLM-L6-v2` model and inserts them into a FAISS index along with their `atomic_query_id`.


In [1]:
import faiss
import numpy as np
import json
import pickle
from sentence_transformers import SentenceTransformer

# 1. Setup the Model
# We use the recommended 'all-MiniLM-L6-v2' model for sentence embeddings
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
print(f"Loading model: {model_name}...")
model = SentenceTransformer(model_name)
print("Model loaded successfully!")


/home/sahil/Desktop/Projects/MEERA/meera_agent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model: sentence-transformers/all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9133.09it/s]


Model loaded successfully!


In [2]:
# 2. Define the Dataset
# In a real scenario, this would be fetched from your database.
# We are storing pairs of (atomic_query_id, text)
dataset = [
    {"atomic_query_id": "AQ_001", "text": """Provide the RBI directions prescribing permissible calling hours 
and conduct standards for recovery/collection agents engaged by 
banks and NBFCs, with a certified copy of the relevant chapter."""},
    {"atomic_query_id": "AQ_002", "text": """State whether a Regulated Entity is fully liable for harassment, abuse 
or criminal intimidation by its outsourced recovery agents, citing the 
exact governing provision, and whether this liability subsists where 
the entity claims the harassing numbers 'are not related to them"""},
    {"atomic_query_id": "AQ_003", "text": """Provide the number of complaints received by the RBI against the 
recovery agents of Muthoot Finance Ltd. during FY 2024-25 and the 
action taken."""},
    {"atomic_query_id": "AQ_004", "text": """Provide a complete list of all virtual digital assets and 
cryptocurrency wallets seized, frozen or held under the directions or 
custody of the RBI in the last five years. (2) For each asset, provide"""}
]

# Extract texts for embedding
texts = [item["text"] for item in dataset]
query_ids = [item["atomic_query_id"] for item in dataset]

print(f"Loaded {len(dataset)} atomic queries.")


Loaded 4 atomic queries.


In [3]:
# 3. Generate Embeddings
print("Generating embeddings...")
# encode() returns a numpy array of shape (num_samples, embedding_dim)
embeddings = model.encode(texts)

embedding_dim = embeddings.shape[1]
print(f"Generated embeddings of shape: {embeddings.shape}")


Generating embeddings...
Generated embeddings of shape: (4, 384)


In [4]:
# 4. Initialize FAISS Database
# Since FAISS IndexIDMap requires integer IDs natively, we will:
# a) Create an integer ID for each atomic query.
# b) Maintain a mapping dictionary to map the integer ID back to the string atomic_query_id.

# Initialize the flat L2 index
index_flat = faiss.IndexFlatL2(embedding_dim)

# Wrap it in an IndexIDMap to store custom integer IDs
index = faiss.IndexIDMap(index_flat)

# Create integer IDs
int_ids = np.arange(len(embeddings)).astype(np.int64)

# Create the dictionary mapping integer IDs to atomic_query_ids
id_mapping = {int(k): v for k, v in zip(int_ids, query_ids)}

# Add embeddings and their corresponding integer IDs to the index
index.add_with_ids(embeddings, int_ids)

print(f"Total vectors stored in FAISS index: {index.ntotal}")


Total vectors stored in FAISS index: 4


In [5]:
# 5. Save the Index and Mapping
faiss.write_index(index, "atomic_queries.index")

with open("id_mapping.pkl", "wb") as f:
    pickle.dump(id_mapping, f)

print("FAISS index and ID mapping saved to disk successfully!")


FAISS index and ID mapping saved to disk successfully!


In [6]:
# 6. Test the Database (Optional)
# Let's perform a quick search to ensure it works
test_query = "In the last five years, what is the complete list of cryptocurrency wallets and virtual digital assets that have been frozen, seized, or held under the custody or directions of the RBI?"
test_emb = model.encode([test_query])

k = 2 # Top 2 results
distances, indices = index.search(test_emb, k)

print(f"Query: '{test_query}'")
for i, idx in enumerate(indices[0]):
    string_id = id_mapping[idx]
    dist = distances[0][i]
    print(f"Rank {i+1} - ID: {string_id}, Distance: {dist:.4f}")


Query: 'In the last five years, what is the complete list of cryptocurrency wallets and virtual digital assets that have been frozen, seized, or held under the custody or directions of the RBI?'
Rank 1 - ID: AQ_004, Distance: 0.1728
Rank 2 - ID: AQ_001, Distance: 1.2438
